# 07 — Single-gene biomarker discovery and replication (Section 4.10)

In [ ]:
import os
from pathlib import Path

# Folder that holds the downloaded public data; all outputs are written into it.
# Set the environment variable GBM_DATA_DIR, or edit the default below.
BASE = Path(os.environ.get("GBM_DATA_DIR", Path.home() / "Downloads"))
print("Data folder:", BASE)

### TCGA -> CGGA EXPRESSION BIOMARKER ANALYSIS

In [ ]:
# ============================================================
# TCGA -> CGGA EXPRESSION BIOMARKER ANALYSIS
#
# Discovery:
#   TCGA development cohort, n=115
#
# Replication:
#   CGGA external cohort, n=104
#
# Primary replicated criterion:
#   TCGA BH-FDR q < 0.05
#   CGGA nominal p < 0.05
#   same HR direction
#
# Strict:
#   FDR < 0.05 in both cohorts
#   same direction
#
# Exploratory:
#   nominal p < 0.05 in both
#   same direction
# ============================================================

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from lifelines import CoxPHFitter
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")


# ============================================================
# 0. PATHS
# ============================================================

DOWNLOADS = BASE

STAGE_D = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
)

OUT_DIR = (
    DOWNLOADS
    / "TCGA_CGGA_BIOMARKER_ANALYSIS"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SPLIT_FILE = (
    STAGE_D
    / "STAGE_D_LOCKED_SPLIT.csv"
)

SURVIVAL_FILE = (
    STAGE_D
    / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)

TCGA_EXPR_FILE = (
    STAGE_D
    / "STAGE_D_M2_EXPRESSION_RAW.csv"
)

CGGA_EXPR_FILE = (
    DOWNLOADS
    / "CGGA_GBM_EXTERNAL_REWIRING"
    / "CGGA_104_HARMONIZED_TO_GTEX.csv"
)

FULL_MAP_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STRING"
    / "TCGA_GBM_gene_to_STRING_mapping.csv"
)


# locate CGGA clinical file
clinical_candidates = [
    p
    for p in DOWNLOADS.rglob(
        "CGGA.mRNAseq_693_clinical.20200506.txt"
    )
    if p.is_file()
]

if not clinical_candidates:
    raise FileNotFoundError(
        "CGGA clinical file not found."
    )

CGGA_CLINICAL_FILE = clinical_candidates[0]


def checkpoint(cond, msg):
    if cond:
        print(f"[CHECKPOINT PASS] {msg}")
    else:
        print(f"[CHECKPOINT FAIL] {msg}")
        raise AssertionError(msg)


# ============================================================
# 1. LOAD TCGA
# ============================================================

print("=" * 90)
print("EXPRESSION BIOMARKER ANALYSIS")
print("=" * 90)

split_df = pd.read_csv(
    SPLIT_FILE
)

survival = (
    pd.read_csv(
        SURVIVAL_FILE
    )
    .set_index("patient_id")
)

tcga_expr = pd.read_csv(
    TCGA_EXPR_FILE,
    index_col=0
)

split_df["patient_id"] = (
    split_df["patient_id"]
    .astype(str)
)

survival.index = (
    survival.index
    .astype(str)
)

tcga_expr.index = (
    tcga_expr.index
    .astype(str)
)

tcga_expr.columns = (
    tcga_expr.columns
    .astype(str)
)


development_ids = (
    split_df.loc[
        split_df["split"] == "development",
        "patient_id"
    ]
    .tolist()
)

checkpoint(
    len(development_ids) == 115,
    "TCGA discovery cohort = 115 development patients"
)

checkpoint(
    set(development_ids).issubset(
        tcga_expr.index
    ),
    "all TCGA discovery patients present in expression matrix"
)


# ============================================================
# 2. LOAD CGGA
# ============================================================

cgga_expr = pd.read_csv(
    CGGA_EXPR_FILE,
    index_col=0
)

# orientation = genes x patients
cgga_expr.index = (
    cgga_expr.index
    .astype(str)
)

cgga_expr.columns = (
    cgga_expr.columns
    .astype(str)
)


clinical = pd.read_csv(
    CGGA_CLINICAL_FILE,
    sep="\t",
    low_memory=False
)

EVENT_COL = (
    "Censor (alive=0; dead=1)"
)

clinical["Age"] = pd.to_numeric(
    clinical["Age"],
    errors="coerce"
)

clinical["OS"] = pd.to_numeric(
    clinical["OS"],
    errors="coerce"
)

clinical[EVENT_COL] = pd.to_numeric(
    clinical[EVENT_COL],
    errors="coerce"
)


eligible = clinical.loc[
    (clinical["PRS_type"] == "Primary")
    &
    (clinical["Histology"] == "GBM")
    &
    (clinical["Grade"] == "WHO IV")
    &
    (
        clinical["IDH_mutation_status"]
        == "Wildtype"
    )
    &
    (clinical["Age"] >= 18)
    &
    (clinical["OS"] > 0)
    &
    (
        clinical[EVENT_COL]
        .isin([0, 1])
    )
].copy()


eligible["CGGA_ID"] = (
    eligible["CGGA_ID"]
    .astype(str)
)

eligible = (
    eligible
    .set_index("CGGA_ID")
)


checkpoint(
    len(eligible) == 104,
    "CGGA replication cohort = 104"
)

checkpoint(
    int(eligible[EVENT_COL].sum()) == 92,
    "CGGA replication events = 92"
)

checkpoint(
    set(eligible.index)
    ==
    set(cgga_expr.columns),
    "CGGA clinical and expression patients exactly match"
)


# ============================================================
# 3. COMMON EXPRESSION GENES
# ============================================================

tcga_genes = set(
    tcga_expr.columns
)

cgga_genes = set(
    cgga_expr.index
)

common_genes = sorted(
    tcga_genes
    &
    cgga_genes
)

print(
    "\nTCGA expression genes:",
    len(tcga_genes)
)

print(
    "CGGA expression genes:",
    len(cgga_genes)
)

print(
    "Common expression genes:",
    len(common_genes)
)

checkpoint(
    len(common_genes) > 1000,
    "sufficient common expression genes"
)


# ============================================================
# 4. GENE ANNOTATION
# ============================================================

annotation = pd.DataFrame({
    "ensembl_id": common_genes
})

if FULL_MAP_FILE.exists():

    gene_map = pd.read_csv(
        FULL_MAP_FILE,
        low_memory=False
    )

    gene_map["ensembl_id"] = (
        gene_map["ensembl_id"]
        .astype(str)
        .str.split(".")
        .str[0]
    )

    keep_cols = [
        c
        for c in [
            "ensembl_id",
            "gene_name",
            "preferred_name"
        ]
        if c in gene_map.columns
    ]

    gene_map = (
        gene_map[
            keep_cols
        ]
        .drop_duplicates(
            "ensembl_id"
        )
    )

    annotation = (
        annotation
        .merge(
            gene_map,
            on="ensembl_id",
            how="left"
        )
    )


# ============================================================
# 5. UNIVARIATE COX FUNCTION
#
# gene standardized within cohort
# HR = effect per 1-SD increase
# ============================================================

def fit_gene_cox(
    gene_values,
    time,
    event
):

    x = np.asarray(
        gene_values,
        dtype=float
    )

    time = np.asarray(
        time,
        dtype=float
    )

    event = np.asarray(
        event,
        dtype=int
    )

    mask = (
        np.isfinite(x)
        &
        np.isfinite(time)
        &
        np.isfinite(event)
        &
        (time > 0)
    )

    x = x[mask]
    time = time[mask]
    event = event[mask]

    if len(x) < 30:
        return None

    sd = np.std(
        x,
        ddof=1
    )

    if (
        not np.isfinite(sd)
        or sd <= 1e-12
    ):
        return None

    # within-cohort z score
    z = (
        x - np.mean(x)
    ) / sd

    df = pd.DataFrame({
        "time": time,
        "event": event,
        "z": z
    })

    try:

        cph = CoxPHFitter()

        cph.fit(
            df,
            duration_col="time",
            event_col="event",
            formula="z"
        )

        row = cph.summary.loc["z"]

        return {
            "beta":
                float(row["coef"]),

            "HR":
                float(row["exp(coef)"]),

            "CI_low":
                float(
                    row[
                        "exp(coef) lower 95%"
                    ]
                ),

            "CI_high":
                float(
                    row[
                        "exp(coef) upper 95%"
                    ]
                ),

            "p":
                float(row["p"]),

            "n":
                len(df),

            "events":
                int(
                    df["event"].sum()
                )
        }

    except Exception:
        return None


# ============================================================
# 6. TCGA DISCOVERY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "TCGA DISCOVERY — EXPRESSION UNIVARIATE COX"
)

print(
    "=" * 90
)


tcga_results = []

tcga_time = (
    survival.loc[
        development_ids,
        "OS_months"
    ]
    .to_numpy(
        dtype=float
    )
)

tcga_event = (
    survival.loc[
        development_ids,
        "OS_event"
    ]
    .to_numpy(
        dtype=int
    )
)


for i, gene in enumerate(
    common_genes,
    start=1
):

    values = (
        tcga_expr.loc[
            development_ids,
            gene
        ]
        .to_numpy(
            dtype=float
        )
    )

    result = fit_gene_cox(
        values,
        tcga_time,
        tcga_event
    )

    if result is not None:

        result[
            "ensembl_id"
        ] = gene

        tcga_results.append(
            result
        )

    if (
        i % 500 == 0
        or i == len(common_genes)
    ):

        print(
            f"  {i:,}/{len(common_genes):,} genes processed"
        )


tcga_results = pd.DataFrame(
    tcga_results
)


checkpoint(
    len(tcga_results) > 0,
    "TCGA expression Cox models fitted"
)


# BH-FDR
tcga_results["q"] = (
    multipletests(
        tcga_results["p"],
        method="fdr_bh"
    )[1]
)


tcga_results = (
    tcga_results
    .merge(
        annotation,
        on="ensembl_id",
        how="left"
    )
)


tcga_results = (
    tcga_results
    .sort_values(
        "p"
    )
    .reset_index(
        drop=True
    )
)


n_tcga_fdr = int(
    (
        tcga_results["q"]
        < 0.05
    )
    .sum()
)


print(
    "\nTCGA expression genes successfully tested:",
    len(tcga_results)
)

print(
    "TCGA discoveries at BH-FDR < 0.05:",
    n_tcga_fdr
)


# ============================================================
# 7. CGGA REPLICATION — TEST ALL COMMON GENES
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "CGGA REPLICATION"
)

print(
    "=" * 90
)


patient_order = (
    eligible.index
    .tolist()
)

cgga_time = (
    eligible.loc[
        patient_order,
        "OS"
    ]
    .to_numpy(
        dtype=float
    )
)

cgga_event = (
    eligible.loc[
        patient_order,
        EVENT_COL
    ]
    .to_numpy(
        dtype=int
    )
)


cgga_results = []


tested_genes = (
    tcga_results[
        "ensembl_id"
    ]
    .tolist()
)


for i, gene in enumerate(
    tested_genes,
    start=1
):

    values = (
        cgga_expr.loc[
            gene,
            patient_order
        ]
        .to_numpy(
            dtype=float
        )
    )

    result = fit_gene_cox(
        values,
        cgga_time,
        cgga_event
    )

    if result is not None:

        result[
            "ensembl_id"
        ] = gene

        cgga_results.append(
            result
        )

    if (
        i % 500 == 0
        or i == len(tested_genes)
    ):

        print(
            f"  {i:,}/{len(tested_genes):,} genes processed"
        )


cgga_results = pd.DataFrame(
    cgga_results
)


checkpoint(
    len(cgga_results) > 0,
    "CGGA expression Cox models fitted"
)


cgga_results["q"] = (
    multipletests(
        cgga_results["p"],
        method="fdr_bh"
    )[1]
)


# ============================================================
# 8. MERGE TCGA + CGGA
# ============================================================

combined = (
    tcga_results[
        [
            "ensembl_id",
            "gene_name",
            "preferred_name",
            "HR",
            "CI_low",
            "CI_high",
            "p",
            "q"
        ]
    ]
    .rename(
        columns={
            "HR":
                "TCGA_HR",

            "CI_low":
                "TCGA_CI_low",

            "CI_high":
                "TCGA_CI_high",

            "p":
                "TCGA_p",

            "q":
                "TCGA_q"
        }
    )
    .merge(
        cgga_results[
            [
                "ensembl_id",
                "HR",
                "CI_low",
                "CI_high",
                "p",
                "q"
            ]
        ]
        .rename(
            columns={
                "HR":
                    "CGGA_HR",

                "CI_low":
                    "CGGA_CI_low",

                "CI_high":
                    "CGGA_CI_high",

                "p":
                    "CGGA_p",

                "q":
                    "CGGA_q"
            }
        ),
        on="ensembl_id",
        how="inner"
    )
)


# ============================================================
# 9. DIRECTION AGREEMENT
# ============================================================

combined[
    "same_direction"
] = (
    (
        (combined["TCGA_HR"] > 1)
        &
        (combined["CGGA_HR"] > 1)
    )
    |
    (
        (combined["TCGA_HR"] < 1)
        &
        (combined["CGGA_HR"] < 1)
    )
)


combined[
    "direction"
] = np.where(
    (
        combined["TCGA_HR"] > 1
    )
    &
    (
        combined["CGGA_HR"] > 1
    ),

    "higher expression -> poorer survival",

    np.where(
        (
            combined["TCGA_HR"] < 1
        )
        &
        (
            combined["CGGA_HR"] < 1
        ),

        "higher expression -> longer survival",

        "discordant"
    )
)


# ============================================================
# 10. REPLICATION DEFINITIONS
# ============================================================

primary = combined.loc[
    (combined["TCGA_q"] < 0.05)
    &
    (combined["CGGA_p"] < 0.05)
    &
    (combined["same_direction"])
].copy()


strict = combined.loc[
    (combined["TCGA_q"] < 0.05)
    &
    (combined["CGGA_q"] < 0.05)
    &
    (combined["same_direction"])
].copy()


exploratory = combined.loc[
    (combined["TCGA_p"] < 0.05)
    &
    (combined["CGGA_p"] < 0.05)
    &
    (combined["same_direction"])
].copy()


# sort by discovery evidence
primary = (
    primary
    .sort_values(
        [
            "TCGA_q",
            "CGGA_p"
        ]
    )
)

strict = (
    strict
    .sort_values(
        [
            "TCGA_q",
            "CGGA_q"
        ]
    )
)

exploratory = (
    exploratory
    .sort_values(
        [
            "TCGA_p",
            "CGGA_p"
        ]
    )
)


# ============================================================
# 11. SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "EXPRESSION BIOMARKER SUMMARY"
)

print(
    "=" * 90
)


print(
    "Common genes screened:",
    len(common_genes)
)

print(
    "TCGA FDR discoveries:",
    n_tcga_fdr
)

print(
    "Primary replicated "
    "(TCGA q<0.05 + CGGA p<0.05 + same direction):",
    len(primary)
)

print(
    "Strict replicated "
    "(FDR<0.05 in both + same direction):",
    len(strict)
)

print(
    "Exploratory nominal concordant "
    "(p<0.05 in both + same direction):",
    len(exploratory)
)


cols_show = [
    "ensembl_id",
    "gene_name",
    "preferred_name",

    "TCGA_HR",
    "TCGA_CI_low",
    "TCGA_CI_high",
    "TCGA_p",
    "TCGA_q",

    "CGGA_HR",
    "CGGA_CI_low",
    "CGGA_CI_high",
    "CGGA_p",
    "CGGA_q",

    "direction"
]


print(
    "\nTOP PRIMARY REPLICATED EXPRESSION CANDIDATES:"
)

if len(primary):

    print(
        primary[
            cols_show
        ]
        .head(30)
        .to_string(
            index=False
        )
    )

else:

    print(
        "NONE under the predefined primary replication criterion."
    )


print(
    "\nTOP STRICT REPLICATED EXPRESSION CANDIDATES:"
)

if len(strict):

    print(
        strict[
            cols_show
        ]
        .head(30)
        .to_string(
            index=False
        )
    )

else:

    print(
        "NONE under strict FDR replication."
    )


print(
    "\nTOP EXPLORATORY NOMINAL CONCORDANT CANDIDATES:"
)

if len(exploratory):

    print(
        exploratory[
            cols_show
        ]
        .head(30)
        .to_string(
            index=False
        )
    )

else:

    print(
        "NONE."
    )


# ============================================================
# 12. SAVE
# ============================================================

tcga_results.to_csv(
    OUT_DIR
    / "TCGA_EXPRESSION_UNIVARIATE_COX_ALL.csv",
    index=False
)

cgga_results.to_csv(
    OUT_DIR
    / "CGGA_EXPRESSION_UNIVARIATE_COX_ALL.csv",
    index=False
)

combined.to_csv(
    OUT_DIR
    / "TCGA_CGGA_EXPRESSION_COX_COMBINED.csv",
    index=False
)

primary.to_csv(
    OUT_DIR
    / "EXPRESSION_PRIMARY_REPLICATED_CANDIDATES.csv",
    index=False
)

strict.to_csv(
    OUT_DIR
    / "EXPRESSION_STRICT_REPLICATED_CANDIDATES.csv",
    index=False
)

exploratory.to_csv(
    OUT_DIR
    / "EXPRESSION_EXPLORATORY_NOMINAL_CANDIDATES.csv",
    index=False
)


print(
    "\nSaved to:"
)

print(
    OUT_DIR
)

print(
    "\nDONE — EXPRESSION BIOMARKER SCREEN COMPLETE."
)

In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd

from lifelines import CoxPHFitter
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# ==========================================================================================
# FINAL FROZEN REWIRING / eNESH BIOMARKER SCREEN
#
# Discovery:
#   TCGA development cohort only, n=115
#
# External replication:
#   CGGA, n=104
#
# IMPORTANT:
#   - Start from all 3,944 externally measurable/common eNESH anchors.
#   - Do NOT prefilter based on CGGA variability before TCGA discovery.
#   - Standardize independently within each cohort.
#   - BH-FDR separately within TCGA and CGGA successful tests.
# ==========================================================================================


# ==========================================================================================
# 1. PATHS
# ==========================================================================================

BASE = BASE

TCGA_ENESH_FILE = (
    BASE
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
    / "STAGE_C_PERSONALIZED_ENESH_144xDEG_ANCHORS.csv"
)

CGGA_ENESH_FILE = (
    BASE
    / "CGGA_GBM_EXTERNAL_REWIRING"
    / "CGGA_104_EXTERNAL_ENESH_3944.csv"
)

OUTDIR = BASE / "TCGA_CGGA_BIOMARKER_ANALYSIS_FINAL_REWIRING"
OUTDIR.mkdir(parents=True, exist_ok=True)


def find_exact(filename):
    matches = list(BASE.rglob(filename))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename}"
        )

    if len(matches) > 1:
        print(f"[WARNING] Multiple copies found for {filename}:")
        for x in matches:
            print("   ", x)
        print("Using:", matches[0])

    return matches[0]


TCGA_SPLIT_FILE = find_exact(
    "STAGE_D_LOCKED_SPLIT.csv"
)

TCGA_SURVIVAL_FILE = find_exact(
    "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)


# Find CGGA clinical file robustly
cgga_clinical_candidates = [
    x for x in BASE.rglob("*")
    if x.is_file()
    and "mrnaseq_693" in x.name.lower()
    and "clinical" in x.name.lower()
]

if not cgga_clinical_candidates:
    raise FileNotFoundError(
        "Could not locate mRNAseq_693 clinical file."
    )

CGGA_CLINICAL_FILE = cgga_clinical_candidates[0]

print("TCGA split:", TCGA_SPLIT_FILE)
print("TCGA survival:", TCGA_SURVIVAL_FILE)
print("CGGA clinical:", CGGA_CLINICAL_FILE)
print("TCGA eNESH:", TCGA_ENESH_FILE)
print("CGGA eNESH:", CGGA_ENESH_FILE)


# ==========================================================================================
# 2. HELPERS
# ==========================================================================================

def smart_read(path):
    try:
        return pd.read_csv(path, sep=None, engine="python")
    except Exception:
        return pd.read_csv(path)


def clean_name(x):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(x).lower()
    )


def find_column(df, candidates, required=True):
    lookup = {
        clean_name(c): c
        for c in df.columns
    }

    # exact normalized match
    for candidate in candidates:
        key = clean_name(candidate)

        if key in lookup:
            return lookup[key]

    # partial match
    for candidate in candidates:
        key = clean_name(candidate)

        for k, original in lookup.items():
            if key in k or k in key:
                return original

    if required:
        raise ValueError(
            f"Could not identify column from:\n{candidates}\n\n"
            f"Available columns:\n{list(df.columns)}"
        )

    return None


def normalize_id(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if re.fullmatch(r"\d+\.0", x):
        x = x[:-2]

    x = x.upper()

    if x.startswith("TCGA-"):
        parts = x.split("-")

        if len(parts) >= 3:
            x = "-".join(parts[:3])

    return x


def convert_event(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce"
    )

    unique_numeric = set(
        numeric.dropna().unique()
    )

    if (
        len(unique_numeric) > 0
        and unique_numeric.issubset({0, 1})
    ):
        return numeric.astype(float)

    mapping = {
        "dead": 1,
        "deceased": 1,
        "death": 1,
        "event": 1,
        "yes": 1,
        "true": 1,

        "alive": 0,
        "living": 0,
        "censored": 0,
        "censor": 0,
        "no": 0,
        "false": 0,
    }

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
        .astype(float)
    )


def fit_cox(x, time, event):

    dat = pd.DataFrame({
        "time": pd.to_numeric(
            time,
            errors="coerce"
        ),
        "event": pd.to_numeric(
            event,
            errors="coerce"
        ),
        "x": pd.to_numeric(
            x,
            errors="coerce"
        ),
    }).dropna()

    if len(dat) < 20:
        raise ValueError(
            "fewer than 20 complete observations"
        )

    if dat["event"].sum() < 5:
        raise ValueError(
            "fewer than 5 events"
        )

    sd = dat["x"].std(ddof=0)

    if not np.isfinite(sd):
        raise ValueError(
            "non-finite SD"
        )

    if sd <= 1e-12:
        raise ValueError(
            "constant or near-constant"
        )

    # Independent cohort-specific z standardization
    dat["x"] = (
        dat["x"] - dat["x"].mean()
    ) / sd

    model = CoxPHFitter()

    model.fit(
        dat,
        duration_col="time",
        event_col="event",
        formula="x",
        show_progress=False,
    )

    row = model.summary.loc["x"]

    return {
        "n": len(dat),
        "events": int(dat["event"].sum()),
        "beta": float(row["coef"]),
        "HR": float(row["exp(coef)"]),
        "CI_low": float(
            row["exp(coef) lower 95%"]
        ),
        "CI_high": float(
            row["exp(coef) upper 95%"]
        ),
        "p": float(row["p"]),
    }


def run_screen(
    X,
    clinical,
    cohort_name
):

    results = []
    failures = []

    genes = list(X.columns)

    print("\n" + "=" * 90)
    print(cohort_name)
    print("=" * 90)

    for i, gene in enumerate(
        genes,
        start=1
    ):

        try:
            out = fit_cox(
                X[gene],
                clinical.loc[
                    X.index,
                    "_TIME_"
                ],
                clinical.loc[
                    X.index,
                    "_EVENT_"
                ],
            )

            out["ensembl_id"] = gene

            results.append(out)

        except Exception as exc:

            failures.append({
                "ensembl_id": gene,
                "reason": str(exc),
            })

        if (
            i % 500 == 0
            or i == len(genes)
        ):
            print(
                f"  {i:,}/{len(genes):,} "
                "anchors processed"
            )

    results = pd.DataFrame(results)
    failures = pd.DataFrame(failures)

    if len(results) > 0:

        results["q"] = multipletests(
            results["p"].values,
            method="fdr_bh"
        )[1]

        results = (
            results
            .sort_values(
                ["q", "p"]
            )
            .reset_index(drop=True)
        )

    return results, failures


# ==========================================================================================
# 3. TCGA CLINICAL + DEVELOPMENT SPLIT
# ==========================================================================================

tcga_surv = smart_read(
    TCGA_SURVIVAL_FILE
)

tcga_split = smart_read(
    TCGA_SPLIT_FILE
)

tcga_surv_id = find_column(
    tcga_surv,
    [
        "patient_id",
        "patient",
        "case_id",
        "barcode",
        "bcr_patient_barcode",
        "id",
    ],
)

tcga_split_id = find_column(
    tcga_split,
    [
        "patient_id",
        "patient",
        "case_id",
        "barcode",
        "bcr_patient_barcode",
        "id",
    ],
)

tcga_time_col = find_column(
    tcga_surv,
    [
        "os_days",
        "os_time",
        "overall_survival_days",
        "overall_survival",
        "survival_time",
        "time",
        "os",
    ],
)

tcga_event_col = find_column(
    tcga_surv,
    [
        "event",
        "os_event",
        "death_event",
        "status",
        "vital_status",
    ],
)

split_col = find_column(
    tcga_split,
    [
        "split",
        "set",
        "partition",
        "cohort",
        "group",
    ],
)

tcga_surv["_ID_"] = (
    tcga_surv[tcga_surv_id]
    .map(normalize_id)
)

tcga_surv["_TIME_"] = pd.to_numeric(
    tcga_surv[tcga_time_col],
    errors="coerce"
)

tcga_surv["_EVENT_"] = convert_event(
    tcga_surv[tcga_event_col]
)

tcga_split["_ID_"] = (
    tcga_split[tcga_split_id]
    .map(normalize_id)
)

split_lookup = (
    tcga_split[
        ["_ID_", split_col]
    ]
    .drop_duplicates("_ID_")
)

tcga_surv = tcga_surv.merge(
    split_lookup,
    on="_ID_",
    how="left"
)

development_mask = (
    tcga_surv[split_col]
    .astype(str)
    .str.lower()
    .str.contains(
        "dev|train",
        regex=True,
        na=False
    )
)

tcga_dev = (
    tcga_surv.loc[
        development_mask,
        [
            "_ID_",
            "_TIME_",
            "_EVENT_",
        ]
    ]
    .dropna()
    .drop_duplicates("_ID_")
    .set_index("_ID_")
)

assert len(tcga_dev) == 115, (
    f"Expected TCGA development n=115, "
    f"found {len(tcga_dev)}"
)

print(
    "\n[PASS] TCGA development cohort:",
    len(tcga_dev)
)


# ==========================================================================================
# 4. CGGA CLINICAL
# ==========================================================================================

cgga_clin = smart_read(
    CGGA_CLINICAL_FILE
)

cgga_id_col = find_column(
    cgga_clin,
    [
        "patient_id",
        "patient",
        "sample_id",
        "cgga_id",
        "id",
    ],
)

cgga_time_col = find_column(
    cgga_clin,
    [
        "os_days",
        "os_time",
        "overall_survival_days",
        "overall_survival",
        "survival_time",
        "time",
        "os",
    ],
)

cgga_event_col = find_column(
    cgga_clin,
    [
        "event",
        "os_event",
        "death_event",
        "status",
        "censor",
    ],
)

cgga_clin["_ID_"] = (
    cgga_clin[cgga_id_col]
    .map(normalize_id)
)

cgga_clin["_TIME_"] = pd.to_numeric(
    cgga_clin[cgga_time_col],
    errors="coerce"
)

cgga_clin["_EVENT_"] = convert_event(
    cgga_clin[cgga_event_col]
)

cgga_clin = (
    cgga_clin[
        [
            "_ID_",
            "_TIME_",
            "_EVENT_",
        ]
    ]
    .dropna()
    .drop_duplicates("_ID_")
    .set_index("_ID_")
)


# ==========================================================================================
# 5. LOAD FROZEN eNESH MATRICES
# ==========================================================================================

tcga_enesh = pd.read_csv(
    TCGA_ENESH_FILE
)

cgga_enesh = pd.read_csv(
    CGGA_ENESH_FILE
)

tcga_matrix_id = tcga_enesh.columns[0]
cgga_matrix_id = cgga_enesh.columns[0]

tcga_enesh["_ID_"] = (
    tcga_enesh[tcga_matrix_id]
    .map(normalize_id)
)

cgga_enesh["_ID_"] = (
    cgga_enesh[cgga_matrix_id]
    .map(normalize_id)
)

tcga_enesh = (
    tcga_enesh
    .drop(
        columns=[tcga_matrix_id]
    )
    .set_index("_ID_")
)

cgga_enesh = (
    cgga_enesh
    .drop(
        columns=[cgga_matrix_id]
    )
    .set_index("_ID_")
)

tcga_enesh = tcga_enesh.apply(
    pd.to_numeric,
    errors="coerce"
)

cgga_enesh = cgga_enesh.apply(
    pd.to_numeric,
    errors="coerce"
)


# ==========================================================================================
# 6. ALIGN PATIENTS
# ==========================================================================================

tcga_ids = (
    tcga_dev.index
    .intersection(
        tcga_enesh.index
    )
)

assert len(tcga_ids) == 115

tcga_dev = tcga_dev.loc[
    tcga_ids
]

tcga_enesh = tcga_enesh.loc[
    tcga_ids
]


cgga_ids = (
    cgga_enesh.index
    .intersection(
        cgga_clin.index
    )
)

cgga_enesh = cgga_enesh.loc[
    cgga_ids
]

cgga_clin = cgga_clin.loc[
    cgga_ids
]

assert len(cgga_ids) == 104, (
    f"Expected CGGA n=104, "
    f"found {len(cgga_ids)}"
)

assert int(
    cgga_clin["_EVENT_"].sum()
) == 92, (
    f"Expected 92 events, "
    f"found {cgga_clin['_EVENT_'].sum()}"
)

print(
    "[PASS] CGGA cohort:",
    len(cgga_ids)
)

print(
    "[PASS] CGGA events:",
    int(
        cgga_clin["_EVENT_"].sum()
    )
)


# ==========================================================================================
# 7. COMMON EXTERNALLY MEASURABLE ANCHORS
# ==========================================================================================

common = sorted(
    set(tcga_enesh.columns)
    &
    set(cgga_enesh.columns)
)

print("\nTCGA eNESH anchors:", tcga_enesh.shape[1])
print("CGGA eNESH anchors:", cgga_enesh.shape[1])
print("Common measurable anchors:", len(common))

assert len(common) == 3944, (
    f"Expected 3944 common measurable anchors, "
    f"found {len(common)}"
)

tcga_X = tcga_enesh[
    common
]

cgga_X = cgga_enesh[
    common
]

print(
    "[PASS] Starting screen from all "
    "3,944 common measurable anchors"
)


# ==========================================================================================
# 8. TCGA DISCOVERY
# ==========================================================================================

tcga_results, tcga_failures = run_screen(
    tcga_X,
    tcga_dev,
    "TCGA DISCOVERY — eNESH COX"
)

print(
    "\nTCGA successfully tested:",
    len(tcga_results)
)

print(
    "TCGA failed/unfit:",
    len(tcga_failures)
)

tcga_discoveries = (
    tcga_results[
        tcga_results["q"] < 0.05
    ]
    .copy()
)

print(
    "TCGA BH-FDR < 0.05 discoveries:",
    len(tcga_discoveries)
)


# ==========================================================================================
# 9. CGGA EXTERNAL SCREEN
#
# Run all common anchors to allow:
#   - primary replication
#   - strict FDR comparison
#   - exploratory nominal concordance
# ==========================================================================================

cgga_results, cgga_failures = run_screen(
    cgga_X,
    cgga_clin,
    "CGGA REPLICATION — eNESH COX"
)

print(
    "\nCGGA successfully tested:",
    len(cgga_results)
)

print(
    "CGGA failed/unfit:",
    len(cgga_failures)
)


# ==========================================================================================
# 10. MERGE
# ==========================================================================================

tcga_out = tcga_results.rename(
    columns={
        "n": "TCGA_n",
        "events": "TCGA_events",
        "beta": "TCGA_beta",
        "HR": "TCGA_HR",
        "CI_low": "TCGA_CI_low",
        "CI_high": "TCGA_CI_high",
        "p": "TCGA_p",
        "q": "TCGA_q",
    }
)

cgga_out = cgga_results.rename(
    columns={
        "n": "CGGA_n",
        "events": "CGGA_events",
        "beta": "CGGA_beta",
        "HR": "CGGA_HR",
        "CI_low": "CGGA_CI_low",
        "CI_high": "CGGA_CI_high",
        "p": "CGGA_p",
        "q": "CGGA_q",
    }
)

combined = tcga_out.merge(
    cgga_out,
    on="ensembl_id",
    how="outer"
)


# ==========================================================================================
# 11. REPLICATION FLAGS
# ==========================================================================================

combined["same_direction"] = (
    np.sign(
        combined["TCGA_beta"]
    )
    ==
    np.sign(
        combined["CGGA_beta"]
    )
)

combined["same_direction"] = (
    combined["same_direction"]
    &
    combined["TCGA_beta"].notna()
    &
    combined["CGGA_beta"].notna()
)


combined["TCGA_discovery_FDR"] = (
    combined["TCGA_q"] < 0.05
)

combined["replicated_primary"] = (
    (combined["TCGA_q"] < 0.05)
    &
    (combined["CGGA_p"] < 0.05)
    &
    combined["same_direction"]
)

combined[
    "replicated_strict_both_FDR"
] = (
    (combined["TCGA_q"] < 0.05)
    &
    (combined["CGGA_q"] < 0.05)
    &
    combined["same_direction"]
)

combined[
    "nominal_both_same_direction"
] = (
    (combined["TCGA_p"] < 0.05)
    &
    (combined["CGGA_p"] < 0.05)
    &
    combined["same_direction"]
)


combined["direction"] = np.where(
    combined["same_direction"]
    &
    (combined["TCGA_beta"] > 0),

    "higher rewiring -> poorer survival",

    np.where(
        combined["same_direction"]
        &
        (combined["TCGA_beta"] < 0),

        "higher rewiring -> longer survival",

        "discordant/unavailable"
    )
)


# ==========================================================================================
# 12. FINAL TABLES
# ==========================================================================================

primary = combined[
    combined["replicated_primary"]
].copy()

strict = combined[
    combined[
        "replicated_strict_both_FDR"
    ]
].copy()

exploratory = combined[
    combined[
        "nominal_both_same_direction"
    ]
].copy()


primary = primary.sort_values(
    [
        "TCGA_q",
        "CGGA_p"
    ]
)

strict = strict.sort_values(
    [
        "TCGA_q",
        "CGGA_q"
    ]
)

exploratory = exploratory.sort_values(
    [
        "TCGA_p",
        "CGGA_p"
    ]
)


# ==========================================================================================
# 13. SUMMARY
# ==========================================================================================

print("\n" + "=" * 90)
print("FINAL FROZEN REWIRING BIOMARKER SUMMARY")
print("=" * 90)

print(
    "Externally measurable common anchors:",
    len(common)
)

print(
    "TCGA successfully tested:",
    len(tcga_results)
)

print(
    "TCGA failed/unfit:",
    len(tcga_failures)
)

print(
    "CGGA successfully tested:",
    len(cgga_results)
)

print(
    "CGGA failed/unfit:",
    len(cgga_failures)
)

print(
    "Successfully tested in BOTH cohorts:",
    int(
        (
            combined["TCGA_p"].notna()
            &
            combined["CGGA_p"].notna()
        ).sum()
    )
)

print(
    "TCGA FDR discoveries:",
    int(
        (
            tcga_results["q"] < 0.05
        ).sum()
    )
)

print(
    "Primary replicated:",
    len(primary)
)

print(
    "Strict replicated:",
    len(strict)
)

print(
    "Exploratory nominal concordant:",
    len(exploratory)
)


# ==========================================================================================
# 14. SHOW RESULTS
# ==========================================================================================

display_cols = [
    "ensembl_id",
    "TCGA_HR",
    "TCGA_CI_low",
    "TCGA_CI_high",
    "TCGA_p",
    "TCGA_q",
    "CGGA_HR",
    "CGGA_CI_low",
    "CGGA_CI_high",
    "CGGA_p",
    "CGGA_q",
    "direction",
]


print("\nTOP PRIMARY REPLICATED:")
if len(primary):
    print(
        primary[
            display_cols
        ]
        .head(20)
        .to_string(index=False)
    )
else:
    print("NONE")


print("\nTOP STRICT REPLICATED:")
if len(strict):
    print(
        strict[
            display_cols
        ]
        .head(20)
        .to_string(index=False)
    )
else:
    print("NONE")


print(
    "\nEXPLORATORY NOMINAL "
    "CONCORDANT CANDIDATES:"
)

if len(exploratory):
    print(
        exploratory[
            display_cols
        ]
        .head(30)
        .to_string(index=False)
    )
else:
    print("NONE")


# ==========================================================================================
# 15. SAVE — NEW FOLDER, OLD RESULTS PRESERVED
# ==========================================================================================

tcga_results.to_csv(
    OUTDIR
    / "FINAL_REWIRING_TCGA_ALL_COX.csv",
    index=False
)

cgga_results.to_csv(
    OUTDIR
    / "FINAL_REWIRING_CGGA_ALL_COX.csv",
    index=False
)

combined.to_csv(
    OUTDIR
    / "FINAL_REWIRING_TCGA_CGGA_ALL.csv",
    index=False
)

primary.to_csv(
    OUTDIR
    / "FINAL_REWIRING_PRIMARY_REPLICATED.csv",
    index=False
)

strict.to_csv(
    OUTDIR
    / "FINAL_REWIRING_STRICT_REPLICATED.csv",
    index=False
)

exploratory.to_csv(
    OUTDIR
    / "FINAL_REWIRING_EXPLORATORY_NOMINAL.csv",
    index=False
)

tcga_failures.to_csv(
    OUTDIR
    / "FINAL_REWIRING_TCGA_FAILURES.csv",
    index=False
)

cgga_failures.to_csv(
    OUTDIR
    / "FINAL_REWIRING_CGGA_FAILURES.csv",
    index=False
)


summary = pd.DataFrame({
    "metric": [
        "common_measurable_anchors",
        "TCGA_successfully_tested",
        "TCGA_failed",
        "CGGA_successfully_tested",
        "CGGA_failed",
        "tested_in_both",
        "TCGA_FDR_discoveries",
        "primary_replicated",
        "strict_both_FDR_replicated",
        "exploratory_nominal_both_same_direction",
    ],

    "value": [
        len(common),
        len(tcga_results),
        len(tcga_failures),
        len(cgga_results),
        len(cgga_failures),
        int(
            (
                combined["TCGA_p"].notna()
                &
                combined["CGGA_p"].notna()
            ).sum()
        ),
        int(
            (
                tcga_results["q"] < 0.05
            ).sum()
        ),
        len(primary),
        len(strict),
        len(exploratory),
    ],
})

summary.to_csv(
    OUTDIR
    / "FINAL_REWIRING_SCREEN_SUMMARY.csv",
    index=False
)


print("\nSaved to:")
print(OUTDIR)

print(
    "\nDONE — FINAL FROZEN "
    "REWIRING BIOMARKER SCREEN COMPLETE."
)

### EXPLORATORY REPLICATION OF TCGA DEVELOPMENT eNESH CANDIDATES

In [ ]:
# =============================================================================
# EXPLORATORY REPLICATION OF TCGA DEVELOPMENT eNESH CANDIDATES
#
# Discovery:
#   TCGA development = 115 patients
#
# Validation:
#   TCGA locked test = 29 patients
#   CGGA external    = 104 patients
#
# IMPORTANT:
#   Candidate set is frozen BEFORE looking at validation outcomes.
#   No feature selection is performed using locked TCGA or CGGA.
# =============================================================================

import numpy as np
import pandas as pd
import warnings

from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceWarning
from statsmodels.stats.multitest import multipletests


print("=" * 95)
print("FROZEN eNESH CANDIDATE REPLICATION")
print("=" * 95)


# =============================================================================
# FILES
# =============================================================================

DISCOVERY_FILE = (
    OUTDIR
    / "TCGA_DEVELOPMENT_UNIVARIATE_COX_ALL_3944.csv"
)

TCGA_TEST_FILE = (
    OUTDIR
    / "TCGA_LOCKED_TEST_ENESH_SURVIVAL.csv"
)

CGGA_FILE = (
    OUTDIR
    / "CGGA_EXTERNAL_104_ENESH_SURVIVAL.csv"
)


# =============================================================================
# LOAD
# =============================================================================

discovery = pd.read_csv(DISCOVERY_FILE)

tcga_test = pd.read_csv(TCGA_TEST_FILE)

cgga = pd.read_csv(CGGA_FILE)


print(
    f"\nTCGA locked test : {len(tcga_test)} patients, "
    f"{int(tcga_test['OS_event'].sum())} events"
)

print(
    f"CGGA external    : {len(cgga)} patients, "
    f"{int(cgga['OS_event'].sum())} events"
)


# =============================================================================
# FREEZE EXPLORATORY CANDIDATES
# =============================================================================
#
# Use only TCGA-development results.
#
# IMPORTANT:
# This threshold must not be modified after validation results are seen.
# =============================================================================

candidates = (
    discovery[
        (discovery["status"] == "ok")
        & (discovery["p_value"] < 0.01)
    ]
    .copy()
    .sort_values("p_value")
    .reset_index(drop=True)
)


print("\n" + "=" * 95)
print("FROZEN DISCOVERY SET")
print("=" * 95)

print(
    f"Frozen candidates: {len(candidates)}"
)


if len(candidates) == 0:
    raise ValueError(
        "No nominal p < 0.01 candidates were found."
    )


if len(candidates) != 55:

    print(
        f"[WARNING] Expected 55 candidates from previous output, "
        f"but found {len(candidates)}."
    )

else:

    print(
        "[CHECKPOINT PASS] 55 exploratory candidates frozen."
    )


# =============================================================================
# MAKE SURE ALL CANDIDATES EXIST IN VALIDATION MATRICES
# =============================================================================

genes = candidates["gene"].tolist()


missing_tcga = [
    g for g in genes
    if g not in tcga_test.columns
]

missing_cgga = [
    g for g in genes
    if g not in cgga.columns
]


if missing_tcga:
    raise ValueError(
        f"{len(missing_tcga)} candidates missing from TCGA locked test."
    )

if missing_cgga:
    raise ValueError(
        f"{len(missing_cgga)} candidates missing from CGGA."
    )


print(
    "[CHECKPOINT PASS] All frozen candidates exist in both validation cohorts."
)


# =============================================================================
# VALIDATION COX FUNCTION
# =============================================================================

def validate_gene(
    df,
    gene,
    mean_dev,
    sd_dev,
    time_col,
    event_col,
):
    """
    Fit univariate Cox using DEVELOPMENT mean/SD.

    The feature scaling is frozen from TCGA development:
        z = (x - development_mean) / development_SD

    Thus HR is expressed per 1 TCGA-development SD.
    """

    x = pd.to_numeric(
        df[gene],
        errors="coerce"
    )

    time = pd.to_numeric(
        df[time_col],
        errors="coerce"
    )

    event = pd.to_numeric(
        df[event_col],
        errors="coerce"
    )


    valid = (
        x.notna()
        & time.notna()
        & event.notna()
        & (time > 0)
    )


    x = x.loc[valid].astype(float)
    time = time.loc[valid].astype(float)
    event = event.loc[valid].astype(int)


    n = len(x)
    events = int(event.sum())


    result = {
        "n": n,
        "events": events,
        "beta": np.nan,
        "HR_per_TCGA_SD": np.nan,
        "CI95_low": np.nan,
        "CI95_high": np.nan,
        "p_value": np.nan,
        "status": "not_estimable",
    }


    if (
        n < 10
        or events < 3
        or not np.isfinite(sd_dev)
        or sd_dev <= 1e-12
    ):

        return result


    # -------------------------------------------------------------------------
    # Frozen TCGA-development transformation
    # -------------------------------------------------------------------------

    z = (
        x - mean_dev
    ) / sd_dev


    # Validation cohort can theoretically be constant
    if (
        not np.isfinite(z.std(ddof=1))
        or z.std(ddof=1) <= 1e-12
    ):

        result["status"] = "zero_variance_validation"

        return result


    cox_df = pd.DataFrame(
        {
            "time": time.values,
            "event": event.values,
            "feature_z": z.values,
        }
    )


    try:

        with warnings.catch_warnings():

            warnings.simplefilter(
                "ignore",
                category=ConvergenceWarning
            )

            warnings.simplefilter(
                "ignore",
                category=RuntimeWarning
            )


            cph = CoxPHFitter()

            cph.fit(
                cox_df,
                duration_col="time",
                event_col="event",
                formula="feature_z"
            )


        row = cph.summary.loc[
            "feature_z"
        ]


        beta = float(
            row["coef"]
        )

        hr = float(
            row["exp(coef)"]
        )

        ci_low = float(
            row["exp(coef) lower 95%"]
        )

        ci_high = float(
            row["exp(coef) upper 95%"]
        )

        p = float(
            row["p"]
        )


        # Reject pathological numerical solutions
        if not all(
            np.isfinite(v)
            for v in [
                beta,
                hr,
                ci_low,
                ci_high,
                p,
            ]
        ):

            result["status"] = "nonfinite_fit"

            return result


        # Extreme coefficients generally indicate separation / instability
        if abs(beta) > 10:

            result["status"] = "unstable_extreme_beta"

            return result


        result.update(
            {
                "beta": beta,
                "HR_per_TCGA_SD": hr,
                "CI95_low": ci_low,
                "CI95_high": ci_high,
                "p_value": p,
                "status": "ok",
            }
        )


        return result


    except Exception as e:

        result["status"] = (
            "fit_failed_"
            + type(e).__name__
        )

        return result


# =============================================================================
# RUN ALL 55 CANDIDATES
# =============================================================================

print("\n" + "=" * 95)
print("RUNNING LOCKED TCGA + EXTERNAL CGGA REPLICATION")
print("=" * 95)


rows = []


for i, row in candidates.iterrows():

    gene = row["gene"]

    mean_dev = float(
        row["mean_dev"]
    )

    sd_dev = float(
        row["sd_dev"]
    )

    beta_dev = float(
        row["beta"]
    )


    # -------------------------------------------------------------------------
    # TCGA LOCKED TEST
    # -------------------------------------------------------------------------

    tcga_result = validate_gene(
        df=tcga_test,
        gene=gene,
        mean_dev=mean_dev,
        sd_dev=sd_dev,
        time_col="OS_months",
        event_col="OS_event",
    )


    # -------------------------------------------------------------------------
    # CGGA EXTERNAL
    # -------------------------------------------------------------------------

    cgga_result = validate_gene(
        df=cgga,
        gene=gene,
        mean_dev=mean_dev,
        sd_dev=sd_dev,
        time_col="OS_months",
        event_col="OS_event",
    )


    # -------------------------------------------------------------------------
    # Direction concordance
    # -------------------------------------------------------------------------

    tcga_same_direction = (
        np.isfinite(tcga_result["beta"])
        and np.sign(tcga_result["beta"])
        == np.sign(beta_dev)
    )


    cgga_same_direction = (
        np.isfinite(cgga_result["beta"])
        and np.sign(cgga_result["beta"])
        == np.sign(beta_dev)
    )


    rows.append(
        {
            # -------------------------------------------------------------
            # Discovery
            # -------------------------------------------------------------
            "gene": gene,
            "dev_beta": beta_dev,
            "dev_HR": row["HR_per_SD"],
            "dev_p": row["p_value"],
            "dev_FDR": row["FDR_BH"],
            "dev_mean": mean_dev,
            "dev_sd": sd_dev,

            # -------------------------------------------------------------
            # TCGA locked test
            # -------------------------------------------------------------
            "tcga_test_n": tcga_result["n"],
            "tcga_test_events": tcga_result["events"],
            "tcga_test_beta": tcga_result["beta"],
            "tcga_test_HR": tcga_result["HR_per_TCGA_SD"],
            "tcga_test_CI_low": tcga_result["CI95_low"],
            "tcga_test_CI_high": tcga_result["CI95_high"],
            "tcga_test_p": tcga_result["p_value"],
            "tcga_test_status": tcga_result["status"],
            "tcga_test_same_direction": tcga_same_direction,

            # -------------------------------------------------------------
            # CGGA external
            # -------------------------------------------------------------
            "cgga_n": cgga_result["n"],
            "cgga_events": cgga_result["events"],
            "cgga_beta": cgga_result["beta"],
            "cgga_HR": cgga_result["HR_per_TCGA_SD"],
            "cgga_CI_low": cgga_result["CI95_low"],
            "cgga_CI_high": cgga_result["CI95_high"],
            "cgga_p": cgga_result["p_value"],
            "cgga_status": cgga_result["status"],
            "cgga_same_direction": cgga_same_direction,
        }
    )


    if (
        (i + 1) % 10 == 0
        or (i + 1) == len(candidates)
    ):

        print(
            f"  {i + 1}/{len(candidates)} candidates validated"
        )


replication = pd.DataFrame(
    rows
)


# =============================================================================
# MULTIPLE-TESTING CORRECTION IN VALIDATION
# =============================================================================
#
# Adjust across the frozen 55 features separately in each validation cohort.
# =============================================================================

replication[
    "tcga_test_FDR_BH"
] = np.nan

replication[
    "cgga_FDR_BH"
] = np.nan


# TCGA
valid_tcga = (
    replication["tcga_test_p"]
    .notna()
)

if valid_tcga.sum() > 0:

    replication.loc[
        valid_tcga,
        "tcga_test_FDR_BH"
    ] = multipletests(
        replication.loc[
            valid_tcga,
            "tcga_test_p"
        ],
        method="fdr_bh"
    )[1]


# CGGA
valid_cgga = (
    replication["cgga_p"]
    .notna()
)

if valid_cgga.sum() > 0:

    replication.loc[
        valid_cgga,
        "cgga_FDR_BH"
    ] = multipletests(
        replication.loc[
            valid_cgga,
            "cgga_p"
        ],
        method="fdr_bh"
    )[1]


# =============================================================================
# REPLICATION FLAGS
# =============================================================================

replication[
    "tcga_test_nominal_replication"
] = (
    replication["tcga_test_same_direction"]
    & (replication["tcga_test_p"] < 0.05)
)


replication[
    "cgga_nominal_replication"
] = (
    replication["cgga_same_direction"]
    & (replication["cgga_p"] < 0.05)
)


replication[
    "cgga_FDR05_replication"
] = (
    replication["cgga_same_direction"]
    & (replication["cgga_FDR_BH"] < 0.05)
)


replication[
    "cgga_FDR10_replication"
] = (
    replication["cgga_same_direction"]
    & (replication["cgga_FDR_BH"] < 0.10)
)


# Both cohorts same direction as discovery
replication[
    "direction_consistent_all3"
] = (
    replication["tcga_test_same_direction"]
    & replication["cgga_same_direction"]
)


# Nominal p < 0.05 in BOTH validation cohorts
replication[
    "nominal_replication_both"
] = (
    replication[
        "tcga_test_nominal_replication"
    ]
    & replication[
        "cgga_nominal_replication"
    ]
)


# =============================================================================
# SORT BY EXTERNAL CGGA EVIDENCE
# =============================================================================

replication = (
    replication
    .sort_values(
        [
            "cgga_FDR_BH",
            "cgga_p",
            "dev_p",
        ],
        ascending=True,
        na_position="last",
    )
    .reset_index(drop=True)
)


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 95)
print("REPLICATION SUMMARY")
print("=" * 95)


print(
    "Frozen candidates:",
    len(replication)
)


print(
    "TCGA locked-test estimable:",
    int(
        (
            replication["tcga_test_status"]
            == "ok"
        ).sum()
    )
)


print(
    "CGGA estimable:",
    int(
        (
            replication["cgga_status"]
            == "ok"
        ).sum()
    )
)


print(
    "\nDirection consistent in TCGA locked test:",
    int(
        replication[
            "tcga_test_same_direction"
        ].sum()
    )
)


print(
    "Direction consistent in CGGA:",
    int(
        replication[
            "cgga_same_direction"
        ].sum()
    )
)


print(
    "Direction consistent in BOTH validation cohorts:",
    int(
        replication[
            "direction_consistent_all3"
        ].sum()
    )
)


print(
    "\nTCGA locked test nominal p<0.05 + same direction:",
    int(
        replication[
            "tcga_test_nominal_replication"
        ].sum()
    )
)


print(
    "CGGA nominal p<0.05 + same direction:",
    int(
        replication[
            "cgga_nominal_replication"
        ].sum()
    )
)


print(
    "CGGA FDR<0.05 + same direction:",
    int(
        replication[
            "cgga_FDR05_replication"
        ].sum()
    )
)


print(
    "CGGA FDR<0.10 + same direction:",
    int(
        replication[
            "cgga_FDR10_replication"
        ].sum()
    )
)


print(
    "Nominal p<0.05 + same direction in BOTH:",
    int(
        replication[
            "nominal_replication_both"
        ].sum()
    )
)


# =============================================================================
# DISPLAY TOP EXTERNAL RESULTS
# =============================================================================

show_cols = [
    "gene",

    "dev_HR",
    "dev_p",

    "tcga_test_HR",
    "tcga_test_p",
    "tcga_test_same_direction",

    "cgga_HR",
    "cgga_p",
    "cgga_FDR_BH",
    "cgga_same_direction",

    "direction_consistent_all3",
]


print("\n" + "=" * 95)
print("TOP CGGA EXTERNAL REPLICATION RESULTS")
print("=" * 95)


print(
    replication[
        show_cols
    ]
    .head(30)
    .to_string(
        index=False
    )
)


# =============================================================================
# STRONGEST EXPLORATORY REPLICATION SET
# =============================================================================
#
# Main external replication criterion:
#
#   1. Frozen TCGA-development candidate
#   2. Same effect direction in CGGA
#   3. CGGA BH-FDR < 0.05
#
# TCGA locked-test evidence is reported independently because n=29 is small.
# =============================================================================

external_replicated = (
    replication[
        replication[
            "cgga_FDR05_replication"
        ]
    ]
    .copy()
)


print("\n" + "=" * 95)
print("CGGA FDR<0.05 EXTERNAL REPLICATION SET")
print("=" * 95)


print(
    f"Replicated candidates: "
    f"{len(external_replicated)}"
)


if len(external_replicated) > 0:

    print(
        external_replicated[
            show_cols
        ]
        .to_string(
            index=False
        )
    )

else:

    print(
        "No candidate achieved CGGA FDR < 0.05 "
        "with concordant direction."
    )


# =============================================================================
# THREE-COHORT DIRECTIONALLY CONSISTENT SET
# =============================================================================

direction_consistent = (
    replication[
        replication[
            "direction_consistent_all3"
        ]
    ]
    .copy()
)


# =============================================================================
# SAVE
# =============================================================================

replication.to_csv(
    OUTDIR
    / "ENESH_55_FROZEN_CANDIDATE_REPLICATION.csv",
    index=False
)


external_replicated.to_csv(
    OUTDIR
    / "ENESH_CGGA_FDR05_REPLICATED.csv",
    index=False
)


direction_consistent.to_csv(
    OUTDIR
    / "ENESH_DIRECTION_CONSISTENT_TCGA_TEST_CGGA.csv",
    index=False
)


# =============================================================================
# FINAL CHECKPOINT
# =============================================================================

print("\n" + "=" * 95)
print("[CHECKPOINT PASS] FROZEN CANDIDATE REPLICATION COMPLETED")
print("=" * 95)

print("\nSaved:")

print(
    OUTDIR
    / "ENESH_55_FROZEN_CANDIDATE_REPLICATION.csv"
)

print(
    OUTDIR
    / "ENESH_CGGA_FDR05_REPLICATED.csv"
)

print(
    OUTDIR
    / "ENESH_DIRECTION_CONSISTENT_TCGA_TEST_CGGA.csv"
)

### FINAL eNESH BIOMARKER RESULTS + FIGURES

In [ ]:
# =============================================================================
# FINAL eNESH BIOMARKER RESULTS + FIGURES
#
# NO NEW TRAINING
# NO NEW FEATURE SELECTION
#
# Inputs:
#   1. TCGA discovery results
#   2. Frozen 55-candidate TCGA/CGGA replication results
#
# Outputs:
#   - final summary tables
#   - discovery vs validation effect-size plots
#   - forest plot
#   - final text summary
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# PATHS
# =============================================================================

DOWNLOADS = BASE

OUTDIR = (
    DOWNLOADS
    / "TCGA_CGGA_BIOMARKER_ANALYSIS"
)

FIGDIR = (
    OUTDIR
    / "FINAL_FIGURES"
)

TABLEDIR = (
    OUTDIR
    / "FINAL_TABLES"
)

FIGDIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLEDIR.mkdir(
    parents=True,
    exist_ok=True
)


DISCOVERY_FILE = (
    OUTDIR
    / "TCGA_DEVELOPMENT_UNIVARIATE_COX_ALL_3944.csv"
)

REPLICATION_FILE = (
    OUTDIR
    / "ENESH_55_FROZEN_CANDIDATE_REPLICATION.csv"
)


# =============================================================================
# CHECK INPUTS
# =============================================================================

print("=" * 100)
print("FINAL eNESH BIOMARKER REPORTING")
print("=" * 100)

for name, path in {
    "Discovery": DISCOVERY_FILE,
    "Replication": REPLICATION_FILE,
}.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name} file not found:\n{path}"
        )

    print(f"[FOUND] {name}")
    print(f"        {path}")


# =============================================================================
# LOAD
# =============================================================================

discovery = pd.read_csv(
    DISCOVERY_FILE
)

replication = pd.read_csv(
    REPLICATION_FILE
)


print("\nDiscovery table:")
print(discovery.shape)

print("\nReplication table:")
print(replication.shape)


# =============================================================================
# DISCOVERY SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("1. DISCOVERY SUMMARY")
print("=" * 100)


n_tested = len(discovery)

n_estimable = int(
    discovery["p_value"]
    .notna()
    .sum()
)

n_p05 = int(
    (
        discovery["p_value"] < 0.05
    ).sum()
)

n_p01 = int(
    (
        discovery["p_value"] < 0.01
    ).sum()
)

n_fdr10 = int(
    (
        discovery["FDR_BH"] < 0.10
    ).sum()
)

n_fdr05 = int(
    (
        discovery["FDR_BH"] < 0.05
    ).sum()
)


print(
    f"Features tested       : {n_tested:,}"
)

print(
    f"Features estimable    : {n_estimable:,}"
)

print(
    f"Nominal p < 0.05      : {n_p05:,}"
)

print(
    f"Nominal p < 0.01      : {n_p01:,}"
)

print(
    f"BH-FDR < 0.10         : {n_fdr10:,}"
)

print(
    f"BH-FDR < 0.05         : {n_fdr05:,}"
)


# =============================================================================
# REPLICATION SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("2. REPLICATION SUMMARY")
print("=" * 100)


n_candidates = len(replication)


tcga_estimable = int(
    (
        replication["tcga_test_status"]
        == "ok"
    ).sum()
)


cgga_estimable = int(
    (
        replication["cgga_status"]
        == "ok"
    ).sum()
)


tcga_direction = int(
    replication[
        "tcga_test_same_direction"
    ].fillna(False).sum()
)


cgga_direction = int(
    replication[
        "cgga_same_direction"
    ].fillna(False).sum()
)


both_direction = int(
    replication[
        "direction_consistent_all3"
    ].fillna(False).sum()
)


tcga_nominal = int(
    replication[
        "tcga_test_nominal_replication"
    ].fillna(False).sum()
)


cgga_nominal = int(
    replication[
        "cgga_nominal_replication"
    ].fillna(False).sum()
)


cgga_fdr05 = int(
    replication[
        "cgga_FDR05_replication"
    ].fillna(False).sum()
)


cgga_fdr10 = int(
    replication[
        "cgga_FDR10_replication"
    ].fillna(False).sum()
)


both_nominal = int(
    replication[
        "nominal_replication_both"
    ].fillna(False).sum()
)


print(
    f"Frozen exploratory candidates : {n_candidates}"
)

print(
    f"TCGA locked-test estimable     : {tcga_estimable}"
)

print(
    f"CGGA external estimable        : {cgga_estimable}"
)

print(
    f"\nSame direction in TCGA test    : {tcga_direction}"
)

print(
    f"Same direction in CGGA         : {cgga_direction}"
)

print(
    f"Same direction in all cohorts  : {both_direction}"
)

print(
    f"\nTCGA nominal replication       : {tcga_nominal}"
)

print(
    f"CGGA nominal replication       : {cgga_nominal}"
)

print(
    f"CGGA FDR < 0.10 replication   : {cgga_fdr10}"
)

print(
    f"CGGA FDR < 0.05 replication   : {cgga_fdr05}"
)

print(
    f"Nominal replication in BOTH    : {both_nominal}"
)


# =============================================================================
# TABLE 1 — TOP DISCOVERY FEATURES
# =============================================================================

print("\n" + "=" * 100)
print("3. TOP DISCOVERY FEATURES")
print("=" * 100)


top_discovery = (
    discovery[
        discovery["status"] == "ok"
    ]
    .sort_values(
        "p_value"
    )
    .head(30)
    .copy()
)


top_discovery_table = (
    top_discovery[
        [
            "gene",
            "HR_per_SD",
            "CI95_low",
            "CI95_high",
            "p_value",
            "FDR_BH",
            "direction",
        ]
    ]
    .copy()
)


top_discovery_table.to_csv(
    TABLEDIR
    / "TABLE_TOP30_TCGA_DISCOVERY.csv",
    index=False
)


print(
    top_discovery_table
    .head(15)
    .to_string(index=False)
)


# =============================================================================
# TABLE 2 — COMPLETE 55-CANDIDATE REPLICATION TABLE
# =============================================================================

final_cols = [
    "gene",

    "dev_HR",
    "dev_p",
    "dev_FDR",

    "tcga_test_HR",
    "tcga_test_CI_low",
    "tcga_test_CI_high",
    "tcga_test_p",
    "tcga_test_FDR_BH",
    "tcga_test_same_direction",

    "cgga_HR",
    "cgga_CI_low",
    "cgga_CI_high",
    "cgga_p",
    "cgga_FDR_BH",
    "cgga_same_direction",

    "direction_consistent_all3",
    "tcga_test_nominal_replication",
    "cgga_nominal_replication",
    "nominal_replication_both",
]


available_cols = [
    c
    for c in final_cols
    if c in replication.columns
]


final_replication_table = (
    replication[
        available_cols
    ]
    .copy()
)


final_replication_table.to_csv(
    TABLEDIR
    / "TABLE_ALL55_REPLICATION_RESULTS.csv",
    index=False
)


# =============================================================================
# TABLE 3 — DIRECTIONALLY CONSISTENT FEATURES
# =============================================================================

direction_consistent_table = (
    replication[
        replication[
            "direction_consistent_all3"
        ].fillna(False)
    ]
    .copy()
)


direction_consistent_table = (
    direction_consistent_table
    .sort_values(
        "dev_p"
    )
)


direction_consistent_table.to_csv(
    TABLEDIR
    / "TABLE_DIRECTION_CONSISTENT_ALL_COHORTS.csv",
    index=False
)


print("\n" + "=" * 100)
print("DIRECTIONALLY CONSISTENT IN ALL THREE COHORTS")
print("=" * 100)

print(
    f"N = {len(direction_consistent_table)}"
)


if len(direction_consistent_table) > 0:

    print(
        direction_consistent_table[
            [
                "gene",
                "dev_HR",
                "dev_p",
                "tcga_test_HR",
                "tcga_test_p",
                "cgga_HR",
                "cgga_p",
                "cgga_FDR_BH",
            ]
        ]
        .head(20)
        .to_string(index=False)
    )


# =============================================================================
# TABLE 4 — NOMINAL CGGA REPLICATION
# =============================================================================

cgga_nominal_table = (
    replication[
        replication[
            "cgga_nominal_replication"
        ].fillna(False)
    ]
    .copy()
)


cgga_nominal_table = (
    cgga_nominal_table
    .sort_values(
        "cgga_p"
    )
)


cgga_nominal_table.to_csv(
    TABLEDIR
    / "TABLE_CGGA_NOMINAL_REPLICATION.csv",
    index=False
)


print("\n" + "=" * 100)
print("CGGA NOMINAL REPLICATION")
print("=" * 100)

print(
    f"N = {len(cgga_nominal_table)}"
)


if len(cgga_nominal_table) > 0:

    print(
        cgga_nominal_table[
            [
                "gene",
                "dev_HR",
                "dev_p",
                "cgga_HR",
                "cgga_p",
                "cgga_FDR_BH",
            ]
        ]
        .to_string(index=False)
    )

else:

    print(
        "No candidate achieved nominal p < 0.05 "
        "in CGGA with concordant direction."
    )


# =============================================================================
# TABLE 5 — FDR EXTERNAL REPLICATION
# =============================================================================

cgga_fdr_table = (
    replication[
        replication[
            "cgga_FDR05_replication"
        ].fillna(False)
    ]
    .copy()
)


cgga_fdr_table.to_csv(
    TABLEDIR
    / "TABLE_CGGA_FDR05_REPLICATION.csv",
    index=False
)


print("\n" + "=" * 100)
print("CGGA FDR<0.05 REPLICATION")
print("=" * 100)

print(
    f"N = {len(cgga_fdr_table)}"
)


# =============================================================================
# FIGURE 1
# TCGA DEVELOPMENT BETA vs TCGA LOCKED-TEST BETA
# =============================================================================

plot_df = replication[
    [
        "gene",
        "dev_beta",
        "tcga_test_beta",
        "tcga_test_status",
    ]
].copy()


plot_df = plot_df[
    (plot_df["tcga_test_status"] == "ok")
    & plot_df["dev_beta"].notna()
    & plot_df["tcga_test_beta"].notna()
]


fig, ax = plt.subplots(
    figsize=(7, 7)
)


ax.scatter(
    plot_df["dev_beta"],
    plot_df["tcga_test_beta"],
    alpha=0.75
)


ax.axhline(
    0,
    linewidth=1
)

ax.axvline(
    0,
    linewidth=1
)


min_val = min(
    plot_df["dev_beta"].min(),
    plot_df["tcga_test_beta"].min()
)

max_val = max(
    plot_df["dev_beta"].max(),
    plot_df["tcga_test_beta"].max()
)


ax.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--",
    linewidth=1
)


ax.set_xlabel(
    "TCGA development log(HR)"
)

ax.set_ylabel(
    "TCGA locked-test log(HR)"
)

ax.set_title(
    "eNESH Effect-Size Consistency\n"
    "TCGA Development vs Locked Test"
)


plt.tight_layout()


plt.savefig(
    FIGDIR
    / "FIGURE1_TCGA_DEV_VS_LOCKED_BETA.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# FIGURE 2
# TCGA DEVELOPMENT BETA vs CGGA BETA
# =============================================================================

plot_df = replication[
    [
        "gene",
        "dev_beta",
        "cgga_beta",
        "cgga_status",
    ]
].copy()


plot_df = plot_df[
    (plot_df["cgga_status"] == "ok")
    & plot_df["dev_beta"].notna()
    & plot_df["cgga_beta"].notna()
]


fig, ax = plt.subplots(
    figsize=(7, 7)
)


ax.scatter(
    plot_df["dev_beta"],
    plot_df["cgga_beta"],
    alpha=0.75
)


ax.axhline(
    0,
    linewidth=1
)

ax.axvline(
    0,
    linewidth=1
)


min_val = min(
    plot_df["dev_beta"].min(),
    plot_df["cgga_beta"].min()
)

max_val = max(
    plot_df["dev_beta"].max(),
    plot_df["cgga_beta"].max()
)


ax.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--",
    linewidth=1
)


ax.set_xlabel(
    "TCGA development log(HR)"
)

ax.set_ylabel(
    "CGGA external log(HR)"
)

ax.set_title(
    "External eNESH Effect-Size Consistency\n"
    "TCGA Development vs CGGA"
)


plt.tight_layout()


plt.savefig(
    FIGDIR
    / "FIGURE2_TCGA_DEV_VS_CGGA_BETA.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# FIGURE 3
# FOREST PLOT — TOP 10 FEATURES DEFINED BY TCGA DEVELOPMENT ONLY
#
# IMPORTANT:
# Features are selected for visualization using development p-values,
# NOT CGGA performance.
# =============================================================================

forest_genes = (
    replication
    .sort_values(
        "dev_p"
    )
    .head(10)
    ["gene"]
    .tolist()
)


forest = (
    replication[
        replication["gene"].isin(
            forest_genes
        )
    ]
    .copy()
)


# Preserve discovery ranking
forest["order"] = (
    forest["gene"]
    .map(
        {
            g: i
            for i, g in enumerate(
                forest_genes
            )
        }
    )
)


forest = forest.sort_values(
    "order"
)


# =============================================================================
# LONG FORMAT
# =============================================================================

forest_rows = []


for _, row in forest.iterrows():

    # TCGA DEVELOPMENT
    forest_rows.append(
        {
            "gene": row["gene"],
            "cohort": "TCGA development",
            "HR": row["dev_HR"],
            "low": np.nan,
            "high": np.nan,
        }
    )

    # We can recover development CI from discovery table
    disc_row = discovery[
        discovery["gene"]
        == row["gene"]
    ]

    if len(disc_row) == 1:

        forest_rows[-1]["low"] = (
            disc_row.iloc[0][
                "CI95_low"
            ]
        )

        forest_rows[-1]["high"] = (
            disc_row.iloc[0][
                "CI95_high"
            ]
        )


    # TCGA LOCKED
    forest_rows.append(
        {
            "gene": row["gene"],
            "cohort": "TCGA locked test",
            "HR": row["tcga_test_HR"],
            "low": row["tcga_test_CI_low"],
            "high": row["tcga_test_CI_high"],
        }
    )


    # CGGA
    forest_rows.append(
        {
            "gene": row["gene"],
            "cohort": "CGGA external",
            "HR": row["cgga_HR"],
            "low": row["cgga_CI_low"],
            "high": row["cgga_CI_high"],
        }
    )


forest_long = pd.DataFrame(
    forest_rows
)


forest_long.to_csv(
    TABLEDIR
    / "TABLE_FOREST_TOP10_VALUES.csv",
    index=False
)


# =============================================================================
# FOREST PLOT
# =============================================================================

fig, ax = plt.subplots(
    figsize=(10, 11)
)


cohorts = [
    "TCGA development",
    "TCGA locked test",
    "CGGA external",
]


offsets = {
    "TCGA development": -0.22,
    "TCGA locked test": 0.00,
    "CGGA external": 0.22,
}


markers = {
    "TCGA development": "o",
    "TCGA locked test": "s",
    "CGGA external": "^",
}


gene_positions = {
    gene: i
    for i, gene in enumerate(
        forest_genes[::-1]
    )
}


for cohort in cohorts:

    subset = forest_long[
        forest_long["cohort"]
        == cohort
    ]


    for _, row in subset.iterrows():

        if not np.isfinite(
            row["HR"]
        ):
            continue


        y = (
            gene_positions[
                row["gene"]
            ]
            + offsets[cohort]
        )


        if (
            np.isfinite(row["low"])
            and np.isfinite(row["high"])
        ):

            xerr = np.array(
                [
                    [
                        row["HR"]
                        - row["low"]
                    ],
                    [
                        row["high"]
                        - row["HR"]
                    ],
                ]
            )


            ax.errorbar(
                row["HR"],
                y,
                xerr=xerr,
                fmt=markers[cohort],
                capsize=3,
                label=(
                    cohort
                    if row["gene"]
                    == forest_genes[0]
                    else None
                )
            )

        else:

            ax.scatter(
                row["HR"],
                y,
                marker=markers[cohort],
                label=(
                    cohort
                    if row["gene"]
                    == forest_genes[0]
                    else None
                )
            )


ax.axvline(
    1.0,
    linestyle="--",
    linewidth=1
)


ax.set_yticks(
    range(
        len(forest_genes)
    )
)

ax.set_yticklabels(
    forest_genes[::-1]
)


ax.set_xscale(
    "log"
)


ax.set_xlabel(
    "Hazard ratio per TCGA-development SD"
)


ax.set_title(
    "Top 10 eNESH Discovery Signals Across Cohorts"
)


ax.legend(
    loc="best"
)


plt.tight_layout()


plt.savefig(
    FIGDIR
    / "FIGURE3_TOP10_ENESH_FOREST_PLOT.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# FIGURE 4
# -log10(p) DISCOVERY vs EXTERNAL CGGA
# =============================================================================

plot_df = replication[
    [
        "gene",
        "dev_p",
        "cgga_p",
        "cgga_same_direction",
    ]
].copy()


plot_df = plot_df[
    plot_df["dev_p"].notna()
    & plot_df["cgga_p"].notna()
]


plot_df["dev_minus_log10_p"] = (
    -np.log10(
        plot_df["dev_p"]
    )
)


plot_df["cgga_minus_log10_p"] = (
    -np.log10(
        plot_df["cgga_p"]
    )
)


fig, ax = plt.subplots(
    figsize=(7, 7)
)


ax.scatter(
    plot_df["dev_minus_log10_p"],
    plot_df["cgga_minus_log10_p"],
    alpha=0.75
)


ax.axhline(
    -np.log10(0.05),
    linestyle="--",
    linewidth=1
)


ax.set_xlabel(
    "-log10(p), TCGA development"
)

ax.set_ylabel(
    "-log10(p), CGGA external"
)

ax.set_title(
    "Discovery vs External Statistical Evidence"
)


plt.tight_layout()


plt.savefig(
    FIGDIR
    / "FIGURE4_DISCOVERY_VS_CGGA_PVALUES.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# FINAL RESULTS SUMMARY FILE
# =============================================================================

summary_lines = []


summary_lines.append(
    "FINAL eNESH BIOMARKER ANALYSIS SUMMARY"
)

summary_lines.append(
    "=" * 70
)


summary_lines.append(
    ""
)

summary_lines.append(
    "DISCOVERY COHORT:"
)

summary_lines.append(
    "TCGA development: 115 patients, 89 events."
)

summary_lines.append(
    f"{n_tested:,} common eNESH features were screened."
)

summary_lines.append(
    f"{n_estimable:,} features yielded estimable univariate Cox models."
)

summary_lines.append(
    f"{n_p01} features had nominal p < 0.01."
)

summary_lines.append(
    f"{n_p05} features had nominal p < 0.05."
)

summary_lines.append(
    f"{n_fdr05} features reached BH-FDR < 0.05."
)


summary_lines.append(
    ""
)

summary_lines.append(
    "VALIDATION:"
)

summary_lines.append(
    "TCGA locked test: 29 patients, 22 events."
)

summary_lines.append(
    "CGGA external cohort: 104 patients, 92 events."
)

summary_lines.append(
    f"{n_candidates} nominal p < 0.01 discovery candidates "
    "were frozen before validation."
)

summary_lines.append(
    f"{tcga_direction} showed concordant direction in the "
    "TCGA locked test."
)

summary_lines.append(
    f"{cgga_direction} showed concordant direction in CGGA."
)

summary_lines.append(
    f"{both_direction} showed concordant direction across "
    "development, locked test, and CGGA."
)

summary_lines.append(
    f"{cgga_nominal} candidates showed nominal p < 0.05 "
    "with concordant direction in CGGA."
)

summary_lines.append(
    f"{cgga_fdr05} candidates achieved BH-FDR < 0.05 "
    "with concordant direction in CGGA."
)


summary_lines.append(
    ""
)

summary_lines.append(
    "INTERPRETATION:"
)

summary_lines.append(
    "The eNESH analysis identified exploratory survival-associated "
    "signals in TCGA development, but no individual eNESH feature "
    "met the prespecified FDR threshold in discovery."
)

summary_lines.append(
    "The frozen exploratory candidates were subsequently evaluated "
    "without reselection in the TCGA locked test and independent "
    "CGGA cohort."
)

summary_lines.append(
    "No individual candidate achieved FDR-significant external "
    "replication in CGGA."
)

summary_lines.append(
    "Therefore, these results support exploratory network-rewiring "
    "associations rather than a formally validated individual "
    "eNESH prognostic biomarker."
)


SUMMARY_FILE = (
    OUTDIR
    / "FINAL_ENESH_BIOMARKER_ANALYSIS_SUMMARY.txt"
)


with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# =============================================================================
# FINAL CONSOLE SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("FINAL CONCLUSION")
print("=" * 100)


print(
    "\nTCGA development:"
)

print(
    f"  {n_p01} exploratory features at p < 0.01"
)

print(
    f"  {n_fdr05} features at FDR < 0.05"
)


print(
    "\nValidation:"
)

print(
    f"  Direction-consistent in all three cohorts: "
    f"{both_direction}"
)

print(
    f"  Nominal CGGA replications: "
    f"{cgga_nominal}"
)

print(
    f"  FDR-significant CGGA replications: "
    f"{cgga_fdr05}"
)


print("\nScientific interpretation:")

print(
    "  eNESH contains exploratory prognostic signals, "
    "but no individual feature currently satisfies "
    "FDR-controlled external biomarker replication."
)


# =============================================================================
# FILE LIST
# =============================================================================

print("\n" + "=" * 100)
print("FILES CREATED")
print("=" * 100)


print("\nTABLES:")

for path in sorted(
    TABLEDIR.glob("*.csv")
):

    print(
        f"  {path.name}"
    )


print("\nFIGURES:")

for path in sorted(
    FIGDIR.glob("*.png")
):

    print(
        f"  {path.name}"
    )


print("\nSUMMARY:")

print(
    f"  {SUMMARY_FILE.name}"
)


print("\nOutput directory:")

print(
    OUTDIR
)


print("\n" + "=" * 100)
print("[CHECKPOINT PASS] FINAL eNESH REPORTING COMPLETED")
print("=" * 100)
